# **FIX CODE PREPROCESSING PART 1 : MEDIAPIPE**

# I. Installing the environment

In [2]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 48.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [3]:
!pip uninstall -y mediapipe numpy scipy
!pip install numpy==1.26.4 mediapipe==0.10.14 pandas==2.1.4

Found existing installation: mediapipe 0.10.35
Uninstalling mediapipe-0.10.35:
  Successfully uninstalled mediapipe-0.10.35
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 845.9 kB/s eta 0:00:00 0:00:01
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0

In [ ]:
import os
# to restart the kernel
os._exit(0)

In [1]:
import numpy as np
import mediapipe as mp
import pandas as pd
print(np.__version__) # 1.26.4
print(pd.__version__)
print(mp.solutions.pose) 

2026-05-31 11:43:00.009831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780227780.271970     126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780227780.350561     126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780227780.985215     126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780227780.985289     126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780227780.985292     126 computation_placer.cc:177] computation placer alr

1.26.4
2.1.4
<module 'mediapipe.python.solutions.pose' from '/usr/local/lib/python3.12/dist-packages/mediapipe/python/solutions/pose.py'>


In [2]:
#for image input output operations
import matplotlib.pyplot as plt
import matplotlib.image as pimg
import sys
import importlib

# II. Start the data preprocessing

* # Knee Angle - Left side

In [ ]:
import cv2
import warnings
import os
import glob

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

#change the folder path and folder target
video_folder_path = '/kaggle/input/datasets/salsaaghnia/mytadataset/071-080/071-080'

FOLDER_TARGET = 'RAW_071_080' 

video_files = sorted(glob.glob(os.path.join(video_folder_path, '*_LEFT*.mp4')))

print(f"the {len(video_files)} has found")

output_raw_folder = f'/kaggle/working/raw_csv_{FOLDER_TARGET}/'
os.makedirs(output_raw_folder, exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity = 2, min_detection_confidence=0.7, min_tracking_confidence=0.7)

#batch processing loop
for i, video_path in enumerate(video_files):
    video_filename = os.path.basename(video_path).replace('.mp4', '')
    print(f"\n[{i+1}/{len(video_files)}] processing: {video_filename}")

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"video resolution: width {width} x height {height}")

    frame_count = 0
    data_list = []

    print("Start video processing")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
   
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
        results = pose.process(image_rgb)
    
   
        if results.pose_landmarks:

            #hip point (x1, y1)
            hip_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP].x * width
            hip_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP].y * height
    
            #knee point (x2, y2) - center
            knee_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE].x * width
            knee_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE].y * height
    
            #ankle point (x3, y3)
            ankle_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_ANKLE].x * width
            ankle_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_ANKLE].y * height
        
            print(f"Frame {frame_count}: left hip = ({hip_x:.4f}, {hip_y:.4f}) , left knee = ({knee_x:.4f}, {knee_y:.4f}), left ankle = ({ankle_x:.4f}, {ankle_y:.4f})")

            #calculating angles
    
            # thigh angle to x-line
            thigh_angle = np.arctan2(hip_y - knee_y, hip_x - knee_x)
    
            # calf angle to x-line
            calf_angle = np.arctan2(ankle_y - knee_y, ankle_x - knee_x)
    
            # angle difference (rad)
            angle_dif = calf_angle - thigh_angle
    
            # conversion to degree
            knee_angle = np.abs(angle_dif * 180.0 / np.pi)
    
            # angle normalization (below 180 degrees)
            if knee_angle > 180.0:
                knee_angle = 360.0 - knee_angle
        
            print(f"Frame {frame_count}: left knee angle = {knee_angle:.2f} degree")
        
        else:
    
            print(f"Frame {frame_count}: no subject found")

            knee_angle = np.nan 
            print(f"Frame {frame_count}: left knee angle = {knee_angle}")

        data_list.append({
        'frame': frame_count,
        'knee_angle': knee_angle})

    cap.release()

    #export files to csv

    df_raw = pd.DataFrame(data_list)
    csv_out_path = os.path.join(output_raw_folder, f"{video_filename}_RAW.csv")
    df_raw.to_csv(csv_out_path, index=False)
    print(f"saved in {csv_out_path}")

pose.close()
print("\n the extraction process is done")

* # Knee Angle - Right side

In [ ]:
import cv2
import warnings
import os
import glob

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

#change the folder path and the folder target
video_folder_path = '/kaggle/input/datasets/salsaaghnia/mytadataset/071-080/071-080'

FOLDER_TARGET = 'RAW_071_080' 

video_files = sorted(glob.glob(os.path.join(video_folder_path, '*_RIGHT*.mp4')))

print(f"the {len(video_files)} has found")

output_raw_folder = f'/kaggle/working/raw_csv_{FOLDER_TARGET}/'
os.makedirs(output_raw_folder, exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity = 2, min_detection_confidence=0.7, min_tracking_confidence=0.7)

#batch processing loop
for i, video_path in enumerate(video_files):
    video_filename = os.path.basename(video_path).replace('.mp4', '')
    print(f"\n[{i+1}/{len(video_files)}] processing: {video_filename}")

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"video resolution: width {width} x height {height}")

    frame_count = 0
    data_list = []

    print("Start video processing")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
   
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
        results = pose.process(image_rgb)
    
   
        if results.pose_landmarks:

            # hip point (x1, y1)
            hip_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_HIP].x * width
            hip_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_HIP].y * height
    
            # knee point (x2, y2) - center
            knee_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_KNEE].x * width
            knee_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_KNEE].y * height
    
            # ankle point (x3, y3)
            ankle_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_ANKLE].x * width
            ankle_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_ANKLE].y * height
        
            # print
            print(f"Frame {frame_count}: right hip = ({hip_x:.4f}, {hip_y:.4f}) , right knee = ({knee_x:.4f}, {knee_y:.4f}), right ankle = ({ankle_x:.4f}, {ankle_y:.4f})")

            #calculating angles
    
            # thigh angle to x-line
            thigh_angle = np.arctan2(hip_y - knee_y, hip_x - knee_x)
    
            # calf angle to x-line
            calf_angle = np.arctan2(ankle_y - knee_y, ankle_x - knee_x)
    
            # angle difference (rad)
            angle_dif = calf_angle - thigh_angle
    
            # conversion to degree
            knee_angle = np.abs(angle_dif * 180.0 / np.pi)
    
            # angle normalization (below 180 degrees)
            if knee_angle > 180.0:
                knee_angle = 360.0 - knee_angle
        
            print(f"Frame {frame_count}: right knee angle = {knee_angle:.2f} degree")
        
        else:
    
            print(f"Frame {frame_count}: no subject found")

            knee_angle = np.nan 
            print(f"Frame {frame_count}: left knee angle = {knee_angle}")

        data_list.append({
        'frame': frame_count,
        'knee_angle': knee_angle})

    cap.release()

    #export files to csv

    df_raw = pd.DataFrame(data_list)
    csv_out_path = os.path.join(output_raw_folder, f"{video_filename}_RAW.csv")
    df_raw.to_csv(csv_out_path, index=False)
    print(f"saved in {csv_out_path}")

pose.close()
print("\n the extraction process is done")

* # Hip Angle - Left Side

In [11]:
import cv2
import warnings
import os
import glob

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

#change the folder path and folder target
video_folder_path = '/kaggle/input/datasets/salsaaghnia/mytadataset/071-080/071-080'

FOLDER_TARGET = 'HIP_RAW_071_080' 

video_files = sorted(glob.glob(os.path.join(video_folder_path, '*_LEFT*.mp4')))

print(f"the {len(video_files)} has found")

output_raw_folder = f'/kaggle/working/raw_csv_{FOLDER_TARGET}/'
os.makedirs(output_raw_folder, exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=2, min_detection_confidence=0.7, min_tracking_confidence=0.7)

#batch processing loop
for i, video_path in enumerate(video_files):
    video_filename = os.path.basename(video_path).replace('.mp4', '')
    print(f"\n[{i+1}/{len(video_files)}] processing: {video_filename}")

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"video resolution: width {width} x height {height}")

    frame_count = 0
    data_list = []

    print("Start video processing")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = pose.process(image_rgb)

        if results.pose_landmarks:

            # hip point (x1, y1) - center of angle
            hip_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP].x * width
            hip_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP].y * height

            # knee point (x2, y2)
            knee_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE].x * width
            knee_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE].y * height

            # virtual vertical point (x3, y3) = (X_Hip, Y_Hip + 1)
            # represents a point directly below the hip along the vertical axis
            vertical_x = hip_x
            vertical_y = hip_y + 1

            print(f"Frame {frame_count}: left hip = ({hip_x:.4f}, {hip_y:.4f}), left knee = ({knee_x:.4f}, {knee_y:.4f}), vertical point = ({vertical_x:.4f}, {vertical_y:.4f})")

            # calculating hip angle
            # vector from hip to knee
            thigh_angle = np.arctan2(knee_y - hip_y, knee_x - hip_x)

            # vector from hip to vertical point (straight down)
            vertical_angle = np.arctan2(vertical_y - hip_y, vertical_x - hip_x)

            # angle difference (rad)
            angle_dif = vertical_angle - thigh_angle

            # conversion to deg
            hip_angle = angle_dif * 180.0 / np.pi

            # angle normalization -180 > x > 180 deg
            if hip_angle > 180.0:
               hip_angle -= 360.0
            elif hip_angle < -180.0:
                hip_angle += 360.0

            print(f"Frame {frame_count}: left hip angle = {hip_angle:.2f} degree")

        else:

            print(f"Frame {frame_count}: no subject found")

            hip_angle = np.nan
            print(f"Frame {frame_count}: left hip angle = {hip_angle}")

        data_list.append({
            'frame': frame_count,
            'hip_angle': hip_angle})

    cap.release()

    # export files to csv
    df_raw = pd.DataFrame(data_list)
    csv_out_path = os.path.join(output_raw_folder, f"{video_filename}_HIP_RAW.csv")
    df_raw.to_csv(csv_out_path, index=False)
    print(f"saved in {csv_out_path}")

pose.close()
print("\n the hip angle extraction process is done")


the 10 has found

[1/10] processing: 071_NM_LEFT
video resolution: width 1920 x height 1080
Start video processing


W0000 00:00:1780236094.855433     537 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780236094.997404     537 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Frame 1: left hip = (1515.7909, 568.4049), left knee = (1435.3273, 735.5322), vertical point = (1515.7909, 569.4049)
Frame 1: left hip angle = -25.71 degree
Frame 2: left hip = (1515.7859, 567.7846), left knee = (1436.0666, 735.2696), vertical point = (1515.7859, 568.7846)
Frame 2: left hip angle = -25.45 degree
Frame 3: left hip = (1497.5202, 568.5984), left knee = (1419.0577, 735.2414), vertical point = (1497.5202, 569.5984)
Frame 3: left hip angle = -25.21 degree
Frame 4: left hip = (1490.5643, 570.9258), left knee = (1414.2949, 735.6010), vertical point = (1490.5643, 571.9258)
Frame 4: left hip angle = -24.85 degree
Frame 5: left hip = (1483.0658, 572.7735), left knee = (1404.9973, 735.6926), vertical point = (1483.0658, 573.7735)
Frame 5: left hip angle = -25.60 degree
Frame 6: left hip = (1463.3746, 572.6425), left knee = (1387.6140, 735.5623), vertical point = (1463.3746, 573.6425)
Frame 6: left hip angle = -24.94 degree
Frame 7: left hip = (1455.7877, 572.6004), left knee = (13

* # Hip Angle - Right Side

In [12]:
import cv2
import warnings
import os
import glob

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

#change the folder path and folder target
video_folder_path = '/kaggle/input/datasets/salsaaghnia/mytadataset/071-080/071-080'

FOLDER_TARGET = 'HIP_RAW_071_080' 

video_files = sorted(glob.glob(os.path.join(video_folder_path, '*_RIGHT*.mp4')))

print(f"the {len(video_files)} has found")

output_raw_folder = f'/kaggle/working/raw_csv_{FOLDER_TARGET}/'
os.makedirs(output_raw_folder, exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=2, min_detection_confidence=0.7, min_tracking_confidence=0.7)

#batch processing loop
for i, video_path in enumerate(video_files):
    video_filename = os.path.basename(video_path).replace('.mp4', '')
    print(f"\n[{i+1}/{len(video_files)}] processing: {video_filename}")

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"video resolution: width {width} x height {height}")

    frame_count = 0
    data_list = []

    print("Start video processing")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = pose.process(image_rgb)

        if results.pose_landmarks:

            # hip point (x1, y1) - center of angle
            hip_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_HIP].x * width
            hip_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_HIP].y * height

            # knee point (x2, y2)
            knee_x = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_KNEE].x * width
            knee_y = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_KNEE].y * height

            # virtual vertical point (x3, y3) = (X_Hip, Y_Hip + 1)
            # represents a point directly below the hip along the vertical axis
            vertical_x = hip_x
            vertical_y = hip_y + 1

            print(f"Frame {frame_count}: right hip = ({hip_x:.4f}, {hip_y:.4f}), right knee = ({knee_x:.4f}, {knee_y:.4f}), vertical point = ({vertical_x:.4f}, {vertical_y:.4f})")

            # calculating hip angle
            # vector from hip to knee
            thigh_angle = np.arctan2(knee_y - hip_y, knee_x - hip_x)

            # vector from hip to vertical point (straight down)
            vertical_angle = np.arctan2(vertical_y - hip_y, vertical_x - hip_x)

            # angle difference (rad)
            angle_dif = thigh_angle - vertical_angle

            # conversion to deg
            hip_angle = angle_dif * 180.0 / np.pi

            # angle normalization -180 > x > 180 deg
            if hip_angle > 180.0:
               hip_angle -= 360.0
            elif hip_angle < -180.0:
                hip_angle += 360.0

            print(f"Frame {frame_count}: right hip angle = {hip_angle:.2f} degree")

        else:

            print(f"Frame {frame_count}: no subject found")

            hip_angle = np.nan
            print(f"Frame {frame_count}: right hip angle = {hip_angle}")

        data_list.append({
            'frame': frame_count,
            'hip_angle': hip_angle})

    cap.release()

    # export files to csv
    df_raw = pd.DataFrame(data_list)
    csv_out_path = os.path.join(output_raw_folder, f"{video_filename}_HIP_RAW.csv")
    df_raw.to_csv(csv_out_path, index=False)
    print(f"saved in {csv_out_path}")

pose.close()
print("\n the hip angle extraction process is done")


the 10 has found

[1/10] processing: 071_NM_RIGHT
video resolution: width 1920 x height 1080
Start video processing


W0000 00:00:1780236346.632436     581 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780236346.769891     581 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Frame 1: right hip = (583.5093, 578.5021), right knee = (661.8793, 736.8361), vertical point = (583.5093, 579.5021)
Frame 1: right hip angle = -26.33 degree
Frame 2: right hip = (586.6825, 578.9474), right knee = (662.1558, 737.9703), vertical point = (586.6825, 579.9474)
Frame 2: right hip angle = -25.39 degree
Frame 3: right hip = (608.4135, 578.9013), right knee = (684.9877, 735.7761), vertical point = (608.4135, 579.9013)
Frame 3: right hip angle = -26.02 degree
Frame 4: right hip = (640.4133, 578.1677), right knee = (719.6587, 731.3017), vertical point = (640.4133, 579.1677)
Frame 4: right hip angle = -27.36 degree
Frame 5: right hip = (646.6969, 577.5787), right knee = (722.2898, 731.6159), vertical point = (646.6969, 578.5787)
Frame 5: right hip angle = -26.14 degree
Frame 6: right hip = (661.2696, 577.0313), right knee = (733.1985, 728.4572), vertical point = (661.2696, 578.0313)
Frame 6: right hip angle = -25.41 degree
Frame 7: right hip = (663.4482, 577.0350), right knee = (7